In [1]:
from datasets import load_dataset
import pandas as pd

# Load the dataset
dataset = load_dataset("nbertagnolli/counsel-chat")

C:\Users\asaf2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


In [2]:
dataset

DatasetDict({
    train: Dataset({
        features: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views'],
        num_rows: 2775
    })
})

In [3]:
df = pd.DataFrame(
    {"question": dataset["train"]["questionText"], "answer": dataset["train"]["answerText"]}
)

In [4]:
df.to_csv("counselchat_english.csv", index=False)


In [19]:
from deep_translator import GoogleTranslator
import pandas as pd

df = pd.read_csv("counselchat_english.csv")
# Initialize the translator
translator = GoogleTranslator(source="en", target="hebrew")

state = dict()
# Function to translate text safely (removing failed translations)
def translate_text(text):
    if isinstance(text, str):
        if text in state:
            return state[text]
        try:
            translated = translator.translate(text)
            if translated:  # Ensure translation is not empty
                state[text] = translated
                return state[text]
        except Exception:
            pass  # Ignore translation failures
    return None  # Mark failed translations

# Load your dataset (assuming df is already loaded)
# df = pd.read_csv("your_file.csv")

# Translate the columns
df["question"] = df["question"].apply(translate_text)
df["answer"] = df["answer"].apply(translate_text)

# Remove rows where any translation failed
df.dropna(subset=["question", "answer"], inplace=True)

# Save the translated file
# Save Hebrew dataset
df.to_csv(r"C:\Users\asaf2\Documents\Projects\Psych Model\counselchat_hebrew3.csv", index=False, encoding="utf-8")

In [20]:
from transformers import (
    MT5Tokenizer,
    MT5ForConditionalGeneration,
    Trainer,
    TrainingArguments,
)
import torch
import pandas as pd

# Load tokenizer and model
model_name = "google/mt5-small"
tokenizer = MT5Tokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name)

import intel_npu_acceleration_library
from intel_npu_acceleration_library.compiler import CompilerConfig

compiler_conf = CompilerConfig(dtype=torch.float32, training=True)
model = intel_npu_acceleration_library.compile(model, compiler_conf)

# Load dataset
df = pd.read_csv(r"C:\Users\asaf2\Documents\Projects\Psych Model\counselchat_hebrew3.csv")
train_data = [{"input": q, "output": a} for q, a in zip(df["question"], df["answer"])]

for item in train_data:
    item["input"] = str(item["input"])
    item["output"] = str(item["output"])


# Tokenize data
def tokenize_data(examples):
    inputs = tokenizer(
        examples["input"], padding="max_length", truncation=True, max_length=256
    )
    outputs = tokenizer(
        examples["output"], padding="max_length", truncation=True, max_length=256
    )
    return {"input_ids": inputs["input_ids"], "labels": outputs["input_ids"]}


from datasets import Dataset

# Create dataset
dataset = Dataset.from_list(train_data).map(tokenize_data)

# Training settings
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    save_steps=10_000,
    save_total_limit=2,
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

# Train the model
trainer.train()

# Save the model
model.save_pretrained(
    r"C:\Users\asaf2\Documents\Projects\Psych Model\hebrew_counseling_model"
)
tokenizer.save_pretrained(
    r"C:\Users\asaf2\Documents\Projects\Psych Model\hebrew_counseling_model"
)
print("Model saved successfully!")


RuntimeError: Failed to import transformers.models.mt5.modeling_mt5 because of the following error (look up to see its traceback):
Failed to import transformers.generation.utils because of the following error (look up to see its traceback):
No module named 'numpy.char'